In [2]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import os



In [3]:
def load_and_combine_model_datasets(base_path: str, model_names: list[str]) -> pd.DataFrame:
    """
    Load CSV files for each model and combine them into a single DataFrame with model labels.
    
    Args:
        base_path: Base directory path containing the CSV files
        model_names: List of model names to process
        
    Returns:
        Combined DataFrame with generator model labels
    """
    all_dfs = []
    
    for model in model_names:
        file_path = os.path.join(base_path, f"{model}_dataset.csv")
        try:
            df = pd.read_csv(file_path, index_col=0)
            # Add generator model name as a column
            df['generator_model'] = model
            all_dfs.append(df)
        except FileNotFoundError:
            print(f"Warning: Could not find dataset for {model} at {file_path}")
    
    # Combine all dataframes
    combined_df = pd.concat(all_dfs, ignore_index=True)
    return combined_df

# Load and combine all datasets
base_path = "./"
model_names = ['gpt-4o', 'deepseek-v3', 'llama_8b', 'llama_70b', 'o1_mini', 'sonnet', 'gpt-4o-mini']

combined_df = load_and_combine_model_datasets(base_path, model_names)

In [4]:
logical_and_df = combined_df[combined_df.question_type == "P_and_Q"]
logical_and_df.head()

,question_triple_id,iteration,hypothesis,topic,reasoning_flaw,question_type,question_title,question_body,avg_forecast,individual_forecasts,consistency_score,generation_reasoning,generator_model
2,iter4_h2_q1,4,The model overestimates the impact of social m...,Sustainable Fashion,NaN,P_and_Q,Will Patagonia's sales of environmentally-frie...,This question resolves as YES if both conditio...,0.75,"[0.75, 0.75, 0.75, 0.75, 0.75]",0.597867,NaN,gpt-4o
3,iter13_h2_q0,13,The model underestimates the impact of geopoli...,Energy and Geopolitics,NaN,P_and_Q,Will China increase its investment in renewabl...,This question resolves as YES if both conditio...,0.41,"[0.45, 0.35, 0.35, 0.45, 0.45]",0.558942,NaN,gpt-4o
6,iter7_h4_q0,7,The model underestimates the impact of geopoli...,Global Energy Markets,NaN,P_and_Q,Will China increase its investment in renewabl...,This question resolves as YES if both conditio...,0.45,"[0.45, 0.45, 0.45, 0.45, 0.45]",0.544915,NaN,gpt-4o
10,iter7_h1_q1,7,The model overestimates the impact of social m...,Sustainable Fashion,NaN,P_and_Q,Will Patagonia's sales of environmentally-frie...,This question resolves as YES if both conditio...,0.57,"[0.55, 0.55, 0.65, 0.45, 0.65]",0.540032,NaN,gpt-4o
12,iter9_h1_q0,9,The model underestimates the impact of geopoli...,Geopolitics and Energy,NaN,P_and_Q,Will China increase its investment in renewabl...,This question resolves as YES if both conditio...,0.43,"[0.35, 0.45, 0.45, 0.45, 0.45]",0.518487,NaN,gpt-4o


In [5]:
# Define topic groupings
topic_to_group = {
    # Energy and Environment
    'energy storage': 'Energy & Environment',
    'economics and energy': 'Energy & Environment',
    'energy': 'Energy & Environment',
    'energy and environment': 'Energy & Environment',
    'energy policy and sustainability': 'Energy & Environment',
    
    # Economics
    'economics': 'Economics',
    'economics and finance': 'Economics',
    
    # Politics
    'geopolitics': 'Politics',
    'economics and education': 'Politics',
    'geopolitics and national security': 'Politics',
    'labor market and technology': 'Politics',
    
    # Computing
    'technology and regulation': 'AI & Computer Science',
    'materials science and quantum computing':  'AI & Computer Science',
    'cryptography':  'AI & Computer Science',
    'ai and computing':  'AI & Computer Science',
    'artificial general intelligence':  'AI & Computer Science',
    'robotics':  'AI & Computer Science',
    'ai and robotics':  'AI & Computer Science',
    'cybersecurity':  'AI & Computer Science',
    
    #Social Media
    'social media and politics': 'Media',
    'social media': 'Media',
    'social media and regulation': 'Media',
    'social media and mental health': 'Media',
    'election misinformation': 'Media',
    'social media regulation':  'Media',
    'marketing and e-commerce': 'Media',
    'elections and misinformation': 'Media',
    
    # Sports
    'sports': 'Sports',
    'sports and olympics': 'Sports'
}

def analyze_model_topics(logical_and_df: pd.DataFrame, model_names: list[str]):
    """
    Analyze and visualize topic distribution between models using MPNet embeddings and t-SNE.
    Uses predefined topic groupings for consistent categorization.
    """
    # Filter for specified models
    model_df = logical_and_df[logical_and_df['generator_model'].isin(model_names)]
    
    # Initialize MPNet model
    mpnet = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
    
    # Get embeddings for questions
    texts = model_df['question_body'].tolist()
    embeddings = mpnet.encode(texts, show_progress_bar=True)
    
    # t-SNE dimensionality reduction
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_embeddings = tsne.fit_transform(embeddings)
    
    # Map topics to groups
    def get_topic_group(topic):
        if pd.isna(topic):
            return 'Other'
        return topic_to_group.get(topic.lower(), 'Other')
    
    # Create visualization DataFrame
    viz_df = pd.DataFrame({
        'TSNE1': tsne_embeddings[:, 0],
        'TSNE2': tsne_embeddings[:, 1],
        'Model': model_df['generator_model'].values,
        'Question': texts,
        'Question_Preview': [t[:100] + '...' for t in texts],
        'Original_Topic': model_df['topic'].values,
        'Topic_Group': model_df['topic'].apply(get_topic_group)
    })
    
    # Define color scheme
    color_map = {
        'Energy & Environment': '#2ecc71',    # Green
        'Economics': '#e74c3c',               # Red
        'Politics': '#3498db',                # Blue
        'Computing': '#9b59b6',               # Purple
        'Artificial General Intelligence': '#f1c40f',  # Yellow
        'Biotechnology': '#1abc9c',           # Turquoise
        'Sports': '#e67e22',                  # Orange
        'Other': '#95a5a6'                    # Grey
    }
    
        # Create interactive plot
    fig = px.scatter(
        viz_df,
        x='TSNE1',
        y='TSNE2',
        color='Topic_Group',
        color_discrete_map=color_map,
        hover_data={
            'TSNE1': False,
            'TSNE2': False,
            'Model': True,
            'Original_Topic': True,
            'Topic_Group': True,
            'Question_Preview': True
        },
        title='Topic Distribution of Adaptive Questions Generated for DeepSeek-V3',
        height=800,
        width=2000  # Compressed width
    )
    
    # Update layout
    fig.update_traces(marker=dict(size=15))
    fig.update_layout(
        template='plotly_white',
        legend=dict(
            title=dict(
                text='Question Topics',
                font=dict(size=32)  # Increased legend title font size
            ),
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=0.02,
            bgcolor='#fdfbf9',
            font=dict(size=29),  # Increased legend text font size
            bordercolor="Black",
            borderwidth=0
        ),
        title=dict(
            text='', #Topic Distribution of Adaptive Questions Generated for DeepSeek-V3
            x=0.35,        # Center title horizontally
            y=0.95,       # Adjust vertical position
            xanchor='center',  # Anchor point for centering
            yanchor='top',    # Anchor point for vertical position
            font=dict(size=20)
        ),
        xaxis=dict(
            title='TSNE1',
            title_font=dict(size=38),  # Increased axis title font size
            tickfont=dict(size=1)     # Increased axis tick font size
        ),
        yaxis=dict(
            title='TSNE2',
            title_font=dict(size=38),  # Increased axis title font size
            tickfont=dict(size=1)     # Increased axis tick font size
        ),
        margin=dict(l=20, r=100, t=50, b=20),
        plot_bgcolor='#fdfbf9'
    )
    
    # Show plot
    fig.show()
    fig.write_image("deepseek_v3_topic_distribution.png", 
                    engine="kaleido",
                    scale=4.0,  # Increases resolution by 4x
                    width=2000,  # Base width
                    height=800)  # Base height
    
    # Print topic distribution
    print("\nTopic group distribution by model:")
    for model in model_names:
        model_data = viz_df[viz_df['Model'] == model]
        topic_dist = model_data['Topic_Group'].value_counts()
        print(f"\n{model} topic distribution:")
        print(topic_dist)
    
    return viz_df

# Run the analysis
model_names = ['deepseek-v3']
viz_df = analyze_model_topics(logical_and_df, model_names)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Topic group distribution by model:

deepseek-v3 topic distribution:
Topic_Group
Other                    15
AI & Computer Science    14
Sports                   11
Energy & Environment      9
Media                     8
Economics                 6
Politics                  4
Name: count, dtype: int64
